# Results Visualization

In [ ]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
from collections import defaultdict

# Plotting
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# Setup
%load_ext autoreload
%autoreload 2

# Data paths
RUN_ID = "20260310_gpt_4_1_x"
RUN_PATH = Path("outputs") / RUN_ID
DATA_PATH = Path("data")

print(f"Loading data from: {RUN_PATH}\n")

In [ ]:
df_sample

## Difficulty vs turns

In [ ]:
from utils.data import load_leetcodedataset_data
import matplotlib.pyplot as plt

df_train, df_test = load_leetcodedataset_data(DATA_PATH)

# Configuration: group size (number of problems per group)
group_size = 10  # Change this to adjust grouping

# Prepare data for stacked plot
df_sample = df_train.head(100).copy()
df_sample['group'] = (df_sample.index // group_size).astype(int)

# Count difficulty levels per group
difficulty_counts = df_sample.groupby(['group', 'difficulty']).size().unstack(fill_value=0)

# Create group labels
group_labels = [f"Problems {i*group_size}-{(i+1)*group_size-1}" for i in range(len(difficulty_counts))]

# Define colors for difficulty levels
colors = {
    'Easy': 'rgba(76, 175, 80, 0.8)',      # Green
    'Medium': 'rgba(255, 193, 7, 0.8)',    # Amber
    'Hard': 'rgba(244, 67, 54, 0.8)'       # Red
}

# Create stacked bar chart
fig = go.Figure()

for difficulty in ['Easy', 'Medium', 'Hard']:
    if difficulty in difficulty_counts.columns:
        fig.add_trace(go.Bar(
            x=group_labels,
            y=difficulty_counts[difficulty],
            name=difficulty.capitalize(),
            marker=dict(color=colors.get(difficulty, 'gray')),
            hovertemplate='<b>%{x}</b><br>' + difficulty.capitalize() + ': %{y}<extra></extra>'
        ))

fig.update_layout(
    title=f"Problem Difficulty Distribution Across Training (Grouped by {group_size} Problems)",
    xaxis_title="Training Progress",
    yaxis_title="Count",
    barmode='stack',
    height=500,
    width=1000,
    plot_bgcolor='rgba(240, 240, 240, 0.5)',
    font=dict(size=12),
    hovermode='x unified'
)

fig.show()

print(f"Difficulty Distribution (group size={group_size}):")
print(difficulty_counts)

## Load Checkpoints Data

In [ ]:
# Load memory (RL state transitions)
memory_path = RUN_PATH / "checkpoints" / "memory.csv"
df_memory = pd.read_csv(memory_path)
print(f"Loaded {len(df_memory)} transitions from memory.csv")
print(f"Columns: {df_memory.columns.tolist()}")
print(f"\nShape: {df_memory.shape}")
print(f"\nFirst few rows:")
df_memory.head()

In [ ]:
# Load interaction files to understand structure
interactions_dir = RUN_PATH / "interactions"
interaction_files = sorted(interactions_dir.glob("problem_*.json"))
print(f"Found {len(interaction_files)} interaction files")

# Load one example to explore structure
with open(interaction_files[0]) as f:
    sample_interaction = json.load(f)

print(f"\nSample interaction (problem 0) has {len(sample_interaction)} entries")
print("Structure:")
for i, entry in enumerate(sample_interaction[:3]):
    print(f"  Entry {i}: {list(entry.keys())}")

# Graph 3: Tutor and Student Abstraction Level Distributions

In [ ]:
def plot_abstraction_level_distribution(df, level_col, title, color_rgb, entity_name):
    """
    Create a bar chart for abstraction level distribution.
    
    Parameters:
    - df: DataFrame
    - level_col: column name (e.g., 'tutor_level', 'student_level')
    - title: chart title
    - color_rgb: color string (e.g., 'rgba(255, 182, 193, 0.8)')
    - entity_name: name of entity (e.g., 'Tutor Level', 'Student Level')
    """
    level_counts = df[level_col].value_counts().sort_index()
    
    fig = go.Figure(data=[
        go.Bar(
            x=level_counts.index.astype(str),
            y=level_counts.values,
            marker=dict(color=color_rgb, line=dict(color='darkred', width=2)),
            text=level_counts.values,
            textposition='auto',
            hovertemplate='<b>Level %{x}</b><br>Count: %{y}<extra></extra>'
        )
    ])
    
    fig.update_layout(
        title=title,
        xaxis_title=f"{entity_name} (1=Concrete, 4=Abstract)",
        yaxis_title="Frequency",
        plot_bgcolor='rgba(240, 240, 240, 0.5)',
        height=500,
        width=700,
        font=dict(size=12),
        showlegend=False
    )
    
    fig.show()
    
    print(f"\n{entity_name} Statistics:")
    print(f"Mean: {df[level_col].mean():.2f}")
    print(f"Median: {df[level_col].median():.2f}")
    print(f"Std Dev: {df[level_col].std():.2f}")
    print(f"\nCounts:\n{level_counts}")

# Plot tutor level distribution
plot_abstraction_level_distribution(
    df_memory,
    'tutor_level',
    'Judged Tutor Abstraction Levels',
    'rgba(255, 182, 193, 0.8)',
    'Tutor Level'
)

In [ ]:
# Plot student level distribution
plot_abstraction_level_distribution(
    df_memory,
    'student_level',
    'Judged Student Abstraction Levels',
    'rgba(135, 206, 250, 0.8)',
    'Student Level'
)

In [ ]:
# Create cross-tabulation (confusion matrix) of student_level vs tutor_level from automated judgments
print(f"Creating confusion matrix from {len(df_memory)} automated transitions\n")

# Create cross-tabulation
confusion_matrix = pd.crosstab(df_memory['student_level'], df_memory['tutor_level'])
print("Student Level vs Tutor Level Cross-Tabulation (Automated Judgments):")
print(confusion_matrix)
print(f"\nShape: {confusion_matrix.shape}")

# Create heatmap
fig = go.Figure(data=go.Heatmap(
    z=confusion_matrix.values,
    x=[f"Tutor Level {i}" for i in confusion_matrix.columns],
    y=[f"Student Level {i}" for i in confusion_matrix.index],
    text=confusion_matrix.values,
    texttemplate='%{text}',
    textfont={"size": 14},
    colorscale='Blues',
    colorbar=dict(title="Count"),
    hovertemplate='<b>%{y}</b><br><b>%{x}</b><br>Count: %{z}<extra></extra>'
))

fig.update_layout(
    title="Judge Alignment: Student Level vs Tutor Level (Automated Judgments)",
    xaxis_title="Tutor Abstraction Level (1=Concrete, 4=Abstract)",
    yaxis_title="Student Abstraction Level (1=Concrete, 4=Abstract)",
    height=600,
    width=750,
    font=dict(size=12),
)

fig.show()

# Print statistics
print("\n" + "="*60)
print("JUDGE ALIGNMENT ANALYSIS")
print("="*60)
print(f"\nMatching levels (where student_level == tutor_level):")
matching = sum(df_memory['student_level'] == df_memory['tutor_level'])
pct = (matching / len(df_memory)) * 100
print(f"  Count: {matching} ({pct:.1f}%)")

print(f"\nStudent ahead of tutor (student_level > tutor_level):")
ahead = sum(df_memory['student_level'] > df_memory['tutor_level'])
pct = (ahead / len(df_memory)) * 100
print(f"  Count: {ahead} ({pct:.1f}%)")

print(f"\nTutor ahead of student (tutor_level > student_level):")
tutor_ahead = sum(df_memory['tutor_level'] > df_memory['student_level'])
pct = (tutor_ahead / len(df_memory)) * 100
print(f"  Count: {tutor_ahead} ({pct:.1f}%)")

print(f"\nLevel difference (abs):")
level_diff = abs(df_memory['student_level'] - df_memory['tutor_level'])
print(f"  Mean: {level_diff.mean():.2f}")
print(f"  Median: {level_diff.median():.2f}")
print(f"  Max: {level_diff.max():.0f}")


# Graph 10: Pedagogical Move Distribution

In [ ]:
# Get pedagogical move distribution
action_counts = df_memory['action'].value_counts()
action_order = ["SOCRATIC_PROBE", "CONCEPTUAL_HINT", "STRUCTURAL_SCAFFOLD"]
action_counts = action_counts.reindex([a for a in action_order if a in action_counts.index])

# Color mapping for actions
action_colors = {
    "SOCRATIC_PROBE": "rgba(100, 149, 237, 0.8)",        # Cornflower blue
    "CONCEPTUAL_HINT": "rgba(144, 238, 144, 0.8)",       # Light green
    "STRUCTURAL_SCAFFOLD": "rgba(255, 165, 0, 0.8)",     # Orange
}

fig = go.Figure(data=[
    go.Bar(
        x=action_counts.index,
        y=action_counts.values,
        marker=dict(
            color=[action_colors.get(action, 'gray') for action in action_counts.index],
            line=dict(color='black', width=1.5)
        ),
        text=action_counts.values,
        textposition='auto',
        hovertemplate='<b>%{x}</b><br>Count: %{y}<extra></extra>'
    )
])

fig.update_layout(
    title="Distribution of Pedagogical Actions (Training Cycle)",
    xaxis_title="Pedagogical Move",
    yaxis_title="Frequency",
    plot_bgcolor='rgba(240, 240, 240, 0.5)',
    height=500,
    width=800,
    font=dict(size=12),
    showlegend=False,
    xaxis=dict(tickangle=-15)
)

fig.show()

print(f"\nPedagogical Move Statistics:")
print(f"Total actions: {action_counts.sum()}")
print(f"\nCounts:")
for action, count in action_counts.items():
    pct = (count / action_counts.sum()) * 100
    print(f"  {action}: {count} ({pct:.1f}%)")

# Graph 10.2: Action with Highest Predicted Reward

In [ ]:
# Extract best actions from predicted rewards
pred_columns = ['pred_reward_SOCRATIC_PROBE', 'pred_reward_CONCEPTUAL_HINT', 'pred_reward_STRUCTURAL_SCAFFOLD']
df_with_preds = df_memory[pred_columns].dropna()

if len(df_with_preds) > 0:
    # Find which action had the highest predicted reward for each row
    best_actions = df_with_preds.idxmax(axis=1).str.replace('pred_reward_', '')
    best_action_counts = best_actions.value_counts()
    best_action_counts = best_action_counts.reindex([a for a in action_order if a in best_action_counts.index])
    
    # Create visualization
    fig = go.Figure(data=[
        go.Bar(
            x=best_action_counts.index,
            y=best_action_counts.values,
            marker=dict(
                color=[action_colors.get(action, 'gray') for action in best_action_counts.index],
                line=dict(color='black', width=1.5)
            ),
            text=best_action_counts.values,
            textposition='auto',
            hovertemplate='<b>%{x}</b><br>Count: %{y}<extra></extra>'
        )
    ])
    
    fig.update_layout(
        title="Distribution of Optimal Actions (Exploitation Cycle)",
        xaxis_title="Pedagogical Move",
        yaxis_title="Frequency",
        plot_bgcolor='rgba(240, 240, 240, 0.5)',
        height=500,
        width=800,
        font=dict(size=12),
        showlegend=False,
        xaxis=dict(tickangle=-15)
    )
    
    fig.show()
    
    # Print statistics
    print(f"\nHighest Predicted Reward Action Statistics:")
    print(f"Rows with predictions: {len(df_with_preds)}")
    print(f"\nCounts:")
    for action, count in best_action_counts.items():
        pct = (count / best_action_counts.sum()) * 100
        print(f"  {action}: {count} ({pct:.1f}%)")
else:
    print("No rows with predicted rewards found")

# Graph 6: Highest Predicted Action Distribution Over Training Time

In [ ]:
pred_columns = ['pred_reward_SOCRATIC_PROBE', 'pred_reward_CONCEPTUAL_HINT', 'pred_reward_STRUCTURAL_SCAFFOLD']
group_size = 20  # Group every N problems

# Extract best actions from predictions
df_with_preds = df_memory[pred_columns].dropna()

if len(df_with_preds) > 0:
    best_actions = df_with_preds.idxmax(axis=1).str.replace('pred_reward_', '')
    problem_nums = df_memory.loc[best_actions.index, 'problem'].values
    
    # Group by training progress
    df_grouped = pd.DataFrame({
        'best_action': best_actions.values,
        'problem': problem_nums
    })
    df_grouped['group'] = (df_grouped['problem'] // group_size).astype(int)
    
    group_action_counts = df_grouped.groupby(['group', 'best_action']).size().unstack(fill_value=0)
    group_action_counts = group_action_counts.reindex([a for a in action_order if a in group_action_counts.columns], axis=1, fill_value=0)
    
    # Create visualization
    group_labels = [f"Problems {i*group_size}-{(i+1)*group_size-1}" for i in group_action_counts.index]
    
    fig = go.Figure()
    for action in group_action_counts.columns:
        fig.add_trace(go.Scatter(
            x=group_labels,
            y=group_action_counts[action],
            mode='lines',
            name=action,
            line=dict(width=0),
            fillcolor=action_colors.get(action, 'gray'),
            stackgroup='one',
            hovertemplate='<b>%{fullData.name}</b><br>%{x}<br>Count: %{y}<extra></extra>'
        ))
    
    fig.update_layout(
        title=f"Distribution of Optimal Actions Over Training (Grouped by {group_size} Problems)",
        xaxis_title="Training Progress",
        yaxis_title="Frequency",
        hovermode='x unified',
        plot_bgcolor='rgba(240, 240, 240, 0.5)',
        height=500,
        width=900,
        font=dict(size=11),
    )
    
    fig.show()
    
    # Print summary
    print(f"\nHighest Predicted Action Over Training Time (group size={group_size}):")
    print(f"Total groups: {len(group_action_counts)}")
    print(f"\n{group_action_counts}")
else:
    print("No rows with predicted rewards found")

# Graph 1: Conversation End Reasons (Pie Chart)

In [ ]:
import json
from collections import Counter

# Extract stop reasons from all interaction files
stop_reasons = []
stop_explanations = Counter()

for interaction_file in interaction_files:
    with open(interaction_file, 'r') as f:
        interactions = json.load(f)
    
    # Find the stop entry (should be last or near end)
    for entry in interactions:
        if isinstance(entry, dict) and "stop" in entry:
            stop_info = entry["stop"]
            
            # Determine the reason
            if "error" in stop_info:
                reason = "Error"
            elif "problem_finished" in stop_info:
                if stop_info["problem_finished"]:
                    reason = "Problem Solved"
                else:
                    explanation = stop_info.get("explanation", "Unknown")
                    reason = explanation
            else:
                reason = "Unknown"
            
            stop_reasons.append(reason)
            stop_explanations[reason] += 1

# Aggregate similar reasons
reason_counts = Counter(stop_reasons)
print("Stop Reasons Distribution:")
for reason, count in reason_counts.most_common():
    pct = (count / len(stop_reasons)) * 100
    print(f"  {reason}: {count} ({pct:.1f}%)")

# Create pie chart
colors = {
    "Problem Solved": "rgba(76, 175, 80, 0.8)",           # Green
    "Max number of iterations": "rgba(255, 193, 7, 0.8)", # Amber
    "Leakage detected": "rgba(244, 67, 54, 0.8)",         # Red
    "Student changed problem": "rgba(233, 30, 99, 0.8)",  # Pink
    "Error": "rgba(156, 39, 176, 0.8)",                   # Purple
}

fig = go.Figure(data=[
    go.Pie(
        labels=list(reason_counts.keys()),
        values=list(reason_counts.values()),
        marker=dict(
            colors=[colors.get(reason, 'rgba(158, 158, 158, 0.8)') for reason in reason_counts.keys()],
            line=dict(color='white', width=2)
        ),
        textposition='inside',
        textinfo='label+percent',
        hovertemplate='<b>%{label}</b><br>Count: %{value}<br>Percentage: %{percent}<extra></extra>'
    )
])

fig.update_layout(
    title="How Conversations Ended (100 Problems)",
    height=600,
    width=700,
    font=dict(size=12),
    showlegend=True
)

fig.show()

print(f"\nTotal problems analyzed: {len(stop_reasons)}")

In [ ]:
# Cumulative Reward vs Turns
# Load memory fresh and compute cumulative metrics per turn
df = pd.read_csv(RUN_PATH / "checkpoints" / "memory.csv")

# Group by turn index (i) and calculate reward statistics
reward_by_turn = df.groupby('i')['reward'].agg(['mean', 'median', 'std', 'count']).reset_index()

print("Cumulative Reward by Turn Index:")
print(reward_by_turn)
print(f"\nTotal turns: {len(reward_by_turn)}")

# Create plot with mean and median lines
fig = go.Figure()

# Mean line
fig.add_trace(go.Scatter(
    x=reward_by_turn['i'],
    y=reward_by_turn['mean'],
    mode='lines+markers',
    name='Mean Reward',
    line=dict(color='rgba(255, 152, 0, 1)', width=3),
    marker=dict(size=8),
    hovertemplate='Turn %{x}: Mean Reward = %{y:.3f}<extra></extra>'
))

# Fill between mean ± std
fig.add_trace(go.Scatter(
    x=reward_by_turn['i'].tolist() + reward_by_turn['i'].tolist()[::-1],
    y=(reward_by_turn['mean'] + reward_by_turn['std']).tolist() + (reward_by_turn['mean'] - reward_by_turn['std']).tolist()[::-1],
    fill='toself',
    fillcolor='rgba(255, 152, 0, 0.2)',
    line=dict(color='rgba(255,255,255,0)'),
    name='±1 Std Dev',
    hoverinfo='skip'
))

fig.update_layout(
    title="RL Reward Signal vs Conversation Turn",
    xaxis_title="Turn Index (0 = first tutor response)",
    yaxis_title="Reward (student_success + pedagogical_quality)",
    height=500,
    width=900,
    plot_bgcolor='rgba(240, 240, 240, 0.5)',
    font=dict(size=12),
    hovermode='x unified',
)

fig.show()

print(f"\nInsights:")
print(f"  Turn 0 mean reward: {reward_by_turn.iloc[0]['mean']:.3f}")
print(f"  Best turn: {reward_by_turn.loc[reward_by_turn['mean'].idxmax(), 'i']:.0f} with {reward_by_turn['mean'].max():.3f}")
print(f"  Worst turn: {reward_by_turn.loc[reward_by_turn['mean'].idxmin(), 'i']:.0f} with {reward_by_turn['mean'].min():.3f}")
print(f"  Overall trend: {'Improving' if reward_by_turn.iloc[-1]['mean'] > reward_by_turn.iloc[0]['mean'] else 'Declining'}")


# Performance Metrics vs Conversation Turns

In [ ]:
# Student Success vs Turns
# Load memory fresh and group by turn index
df = pd.read_csv(RUN_PATH / "checkpoints" / "memory.csv")

# Group by turn index (i) and calculate statistics
success_by_turn = df.groupby('i')['student_success'].agg(['mean', 'median', 'std', 'count']).reset_index()

print("Student Success by Turn Index:")
print(success_by_turn)
print(f"\nTotal turns: {len(success_by_turn)}")

# Create plot with mean and median lines
fig = go.Figure()

# Mean line
fig.add_trace(go.Scatter(
    x=success_by_turn['i'],
    y=success_by_turn['mean'],
    mode='lines+markers',
    name='Mean Success',
    line=dict(color='rgba(76, 175, 80, 1)', width=3),
    marker=dict(size=8),
    hovertemplate='Turn %{x}: Mean Success = %{y:.2%}<extra></extra>'
))

# Fill between mean ± std
fig.add_trace(go.Scatter(
    x=success_by_turn['i'].tolist() + success_by_turn['i'].tolist()[::-1],
    y=(success_by_turn['mean'] + success_by_turn['std']).tolist() + (success_by_turn['mean'] - success_by_turn['std']).tolist()[::-1],
    fill='toself',
    fillcolor='rgba(76, 175, 80, 0.2)',
    line=dict(color='rgba(255,255,255,0)'),
    name='±1 Std Dev',
    hoverinfo='skip'
))

fig.update_layout(
    title="Student Code Success vs Conversation Turn",
    xaxis_title="Turn Index (0 = first tutor response)",
    yaxis_title="Student Success Rate (% tests passed)",
    height=500,
    width=900,
    plot_bgcolor='rgba(240, 240, 240, 0.5)',
    font=dict(size=12),
    hovermode='x unified',
)

fig.update_yaxes(tickformat='.0%')
fig.show()

print(f"\nInsights:")
print(f"  Turn 0 mean success: {success_by_turn.iloc[0]['mean']:.1%}")
print(f"  Best turn: {success_by_turn.loc[success_by_turn['mean'].idxmax(), 'i']:.0f} with {success_by_turn['mean'].max():.1%}")
print(f"  Worst turn: {success_by_turn.loc[success_by_turn['mean'].idxmin(), 'i']:.0f} with {success_by_turn['mean'].min():.1%}")


In [ ]:
# Model Training Progress Across 100 Conversations
# Load memory fresh
df = pd.read_csv(RUN_PATH / "checkpoints" / "memory.csv")

# Configuration: group size (number of consecutive problems to group together)
group_size_problems = 5  # Number of problems to group (configurable)

# Calculate mean accuracy per problem (0-99)
accuracy_by_problem = df.groupby('problem')['student_success'].agg(['mean', 'std', 'count']).reset_index()
accuracy_by_problem.columns = ['problem', 'accuracy_mean', 'accuracy_std', 'count']

# Group problems into chunks and calculate mean accuracy per group
accuracy_by_problem['group'] = (accuracy_by_problem['problem'] // group_size_problems).astype(int)
grouped_accuracy = accuracy_by_problem.groupby('group').agg({
    'accuracy_mean': ['mean', 'std'],
    'count': 'sum'
}).reset_index()

# Flatten column names
grouped_accuracy.columns = ['group', 'mean', 'std', 'total_transitions']

# Create group labels
grouped_accuracy['label'] = grouped_accuracy['group'].apply(
    lambda g: f"Problems {g*group_size_problems}-{(g+1)*group_size_problems-1}"
)

print(f"Model Training Progress Across 100 Conversations (group size = {group_size_problems} problems):")
print(grouped_accuracy[['label', 'mean', 'std', 'total_transitions']])

# Create bar chart with error bars
fig = go.Figure(data=[
    go.Bar(
        x=grouped_accuracy['label'],
        y=grouped_accuracy['mean'],
        error_y=dict(
            type='data',
            array=grouped_accuracy['std'],
            visible=True
        ),
        marker=dict(
            color='rgba(66, 133, 244, 0.8)',
            line=dict(color='rgba(25, 103, 210, 1)', width=2)
        ),
        text=[f"{v:.1%}" for v in grouped_accuracy['mean']],
        textposition='outside',
        hovertemplate='<b>%{x}</b><br>Mean Accuracy: %{y:.2%}<br>Std Dev: %{error_y.array:.2%}<br>Transitions: %{customdata}<extra></extra>',
        customdata=grouped_accuracy['total_transitions']
    )
])

fig.update_layout(
    title=f"Model Training Progress: Student Accuracy Across 100 Conversations (Grouped by {group_size_problems} Problems)",
    xaxis_title="Training Conversation Progress",
    yaxis_title="Mean Student Success Rate",
    height=500,
    width=1000,
    plot_bgcolor='rgba(240, 240, 240, 0.5)',
    font=dict(size=12),
    showlegend=False
)

fig.update_yaxes(tickformat='.0%', range=[0, 1.0])
fig.show()

# Print summary statistics
print(f"\n{'='*60}")
print(f"Training Progress Summary (group size = {group_size_problems} problems):")
print(f"{'='*60}")
print(f"Total groups: {len(grouped_accuracy)}")
print(f"First group (early training) mean accuracy: {grouped_accuracy.iloc[0]['mean']:.1%}")
print(f"Last group (late training) mean accuracy: {grouped_accuracy.iloc[-1]['mean']:.1%}")
improvement = grouped_accuracy.iloc[-1]['mean'] - grouped_accuracy.iloc[0]['mean']
print(f"Overall improvement: {improvement:+.1%}")
best_idx = grouped_accuracy['mean'].idxmax()
print(f"Best group: {grouped_accuracy.loc[best_idx, 'label']} ({grouped_accuracy.loc[best_idx, 'mean']:.1%})")
worst_idx = grouped_accuracy['mean'].idxmin()
print(f"Worst group: {grouped_accuracy.loc[worst_idx, 'label']} ({grouped_accuracy.loc[worst_idx, 'mean']:.1%})")


In [ ]:
# Model Training Progress: Cumulative Reward Across 100 Conversations
# Load memory fresh
df = pd.read_csv(RUN_PATH / "checkpoints" / "memory.csv")

# Configuration: group size (number of consecutive problems to group together)
group_size_problems = 10  # Number of problems to group (configurable)

# Calculate mean reward per problem (0-99)
reward_by_problem = df.groupby('problem')['reward'].agg(['mean', 'std', 'count']).reset_index()
reward_by_problem.columns = ['problem', 'reward_mean', 'reward_std', 'count']

# Group problems into chunks and calculate mean reward per group
reward_by_problem['group'] = (reward_by_problem['problem'] // group_size_problems).astype(int)
grouped_reward = reward_by_problem.groupby('group').agg({
    'reward_mean': ['mean', 'std'],
    'count': 'sum'
}).reset_index()

# Flatten column names
grouped_reward.columns = ['group', 'mean', 'std', 'total_transitions']

# Create group labels
grouped_reward['label'] = grouped_reward['group'].apply(
    lambda g: f"Problems {g*group_size_problems}-{(g+1)*group_size_problems-1}"
)

print(f"Model Training Progress - Cumulative Reward (group size = {group_size_problems} problems):")
print(grouped_reward[['label', 'mean', 'std', 'total_transitions']])

# Create bar chart with error bars
fig = go.Figure(data=[
    go.Bar(
        x=grouped_reward['label'],
        y=grouped_reward['mean'],
        error_y=dict(
            type='data',
            array=grouped_reward['std'],
            visible=True
        ),
        marker=dict(
            color='rgba(255, 152, 0, 0.8)',
            line=dict(color='rgba(230, 124, 0, 1)', width=2)
        ),
        text=[f"{v:.3f}" for v in grouped_reward['mean']],
        textposition='outside',
        hovertemplate='<b>%{x}</b><br>Mean Reward: %{y:.4f}<br>Std Dev: %{error_y.array:.4f}<br>Transitions: %{customdata}<extra></extra>',
        customdata=grouped_reward['total_transitions']
    )
])

fig.update_layout(
    title=f"Model Training Progress: Cumulative Reward Across 100 Conversations (Grouped by {group_size_problems} Problems)",
    xaxis_title="Training Conversation Progress",
    yaxis_title="Mean Reward Signal",
    height=500,
    width=1000,
    plot_bgcolor='rgba(240, 240, 240, 0.5)',
    font=dict(size=12),
    showlegend=False
)

fig.show()

# Print summary statistics
print(f"\n{'='*60}")
print(f"Training Progress Summary - Reward (group size = {group_size_problems} problems):")
print(f"{'='*60}")
print(f"Total groups: {len(grouped_reward)}")
print(f"First group (early training) mean reward: {grouped_reward.iloc[0]['mean']:.4f}")
print(f"Last group (late training) mean reward: {grouped_reward.iloc[-1]['mean']:.4f}")
improvement = grouped_reward.iloc[-1]['mean'] - grouped_reward.iloc[0]['mean']
print(f"Overall change: {improvement:+.4f}")
best_idx = grouped_reward['mean'].idxmax()
print(f"Best group: {grouped_reward.loc[best_idx, 'label']} ({grouped_reward.loc[best_idx, 'mean']:.4f})")
worst_idx = grouped_reward['mean'].idxmin()
print(f"Worst group: {grouped_reward.loc[worst_idx, 'label']} ({grouped_reward.loc[worst_idx, 'mean']:.4f})")

In [ ]:
# Model Training Progress: Reward Model Mean Absolute Error (MAE) Across 100 Conversations
# Load memory fresh
df = pd.read_csv(RUN_PATH / "checkpoints" / "memory.csv")

# Configuration: group size (number of consecutive problems to group together)
group_size_problems = 20  # Number of problems to group (configurable)

# Compute MAE for each transition
# MAE = |actual_reward - predicted_reward|
# For predicted reward, use the best (max) predicted reward among the three actions
pred_columns = ['pred_reward_SOCRATIC_PROBE', 'pred_reward_CONCEPTUAL_HINT', 'pred_reward_STRUCTURAL_SCAFFOLD']
df_with_preds = df[pred_columns + ['reward', 'problem']].dropna()

if len(df_with_preds) > 0:
    # Get best predicted reward for each row
    best_pred_reward = df_with_preds[pred_columns].max(axis=1)
    
    # Compute MAE
    df_with_preds['mae'] = abs(df_with_preds['reward'] - best_pred_reward)
    
    # Calculate mean MAE per problem (0-99)
    mae_by_problem = df_with_preds.groupby('problem')['mae'].agg(['mean', 'std', 'count']).reset_index()
    mae_by_problem.columns = ['problem', 'mae_mean', 'mae_std', 'count']
    
    # Group problems into chunks and calculate mean MAE per group
    mae_by_problem['group'] = (mae_by_problem['problem'] // group_size_problems).astype(int)
    grouped_mae = mae_by_problem.groupby('group').agg({
        'mae_mean': ['mean', 'std'],
        'count': 'sum'
    }).reset_index()
    
    # Flatten column names
    grouped_mae.columns = ['group', 'mean', 'std', 'total_transitions']
    
    # Create group labels
    grouped_mae['label'] = grouped_mae['group'].apply(
        lambda g: f"Problems {g*group_size_problems}-{(g+1)*group_size_problems-1}"
    )
    
    print(f"Model Training Progress - Reward Model MAE (group size = {group_size_problems} problems):")
    print(grouped_mae[['label', 'mean', 'std', 'total_transitions']])
    
    # Create bar chart with error bars
    fig = go.Figure(data=[
        go.Bar(
            x=grouped_mae['label'],
            y=grouped_mae['mean'],
            error_y=dict(
                type='data',
                array=grouped_mae['std'],
                visible=True
            ),
            marker=dict(
                color='rgba(244, 67, 54, 0.8)',
                line=dict(color='rgba(200, 30, 20, 1)', width=2)
            ),
            text=[f"{v:.4f}" for v in grouped_mae['mean']],
            textposition='outside',
            hovertemplate='<b>%{x}</b><br>Mean MAE: %{y:.5f}<br>Std Dev: %{error_y.array:.5f}<br>Transitions: %{customdata}<extra></extra>',
            customdata=grouped_mae['total_transitions']
        )
    ])
    
    fig.update_layout(
        title=f"Model Training Progress: Reward Model MAE Across 100 Conversations (Grouped by {group_size_problems} Problems)",
        xaxis_title="Training Conversation Progress",
        yaxis_title="Mean Absolute Error (Actual vs Predicted Reward)",
        height=500,
        width=1000,
        plot_bgcolor='rgba(240, 240, 240, 0.5)',
        font=dict(size=12),
        showlegend=False
    )
    
    fig.show()
    
    # Print summary statistics
    print(f"\n{'='*60}")
    print(f"Training Progress Summary - MAE (group size = {group_size_problems} problems):")
    print(f"{'='*60}")
    print(f"Total groups: {len(grouped_mae)}")
    print(f"First group (early training) mean MAE: {grouped_mae.iloc[0]['mean']:.5f}")
    print(f"Last group (late training) mean MAE: {grouped_mae.iloc[-1]['mean']:.5f}")
    improvement = grouped_mae.iloc[0]['mean'] - grouped_mae.iloc[-1]['mean']
    print(f"MAE reduction (improvement): {improvement:+.5f}")
    best_idx = grouped_mae['mean'].idxmin()
    print(f"Best group (lowest MAE): {grouped_mae.loc[best_idx, 'label']} ({grouped_mae.loc[best_idx, 'mean']:.5f})")
    worst_idx = grouped_mae['mean'].idxmax()
    print(f"Worst group (highest MAE): {grouped_mae.loc[worst_idx, 'label']} ({grouped_mae.loc[worst_idx, 'mean']:.5f})")
else:
    print("No rows with predicted rewards found")